In [4]:
import pandas as pd
import numpy as np

In [2]:
XY = pd.read_parquet('FX_test.parquet')

In [6]:
Y = np.log(XY[['HG=F']])
X = np.log(XY.drop(columns = ['HG=F']))

In [27]:
def ds(Y, X, k):
    Yk = Y.diff(k)
    Xk = X.diff(k).shift(1)
    data = Yk.join(Xk).dropna()
    return data[Y.columns], data[X.columns]

In [28]:
def expanding_outer_split(
    X,
    Y,
    outer_train=2000,
    outer_test=60,
    step=60,
    h=1,
):
    embargo = h - 1
    n = len(X)
    split_num = 0
    test_start = outer_train

    while test_start + outer_test <= n:
        train_end = test_start - embargo

        if train_end <= 0:
            break
        yield {
            "split": split_num,
            "X_train": X.iloc[:train_end],
            "Y_train": Y.iloc[:train_end],
            "X_test": X.iloc[test_start:test_start + outer_test],
            "Y_test": Y.iloc[test_start:test_start + outer_test],
        }
        split_num += 1
        test_start += step

In [29]:
Y1, X1 = ds(Y, X, 1)
Y5, X5 = ds(Y, X, 5)
Y20, X20 = ds(Y, X, 20)

splits1 = list(expanding_outer_split(X1, Y1, outer_train=2000, outer_test=60, step=60, h=1))
splits5 = list(expanding_outer_split(X5, Y5, outer_train=2000, outer_test=60, step=60, h=5))
splits20 = list(expanding_outer_split(X20, Y20, outer_train=2000, outer_test=60, step=60, h=20))

In [32]:
def ic(y, yhat):
    y = pd.Series(np.asarray(y).ravel())
    yhat = pd.Series(np.asarray(yhat).ravel())
    return y.corr(yhat, method="spearman")

def rmse(y, yhat):
    y = np.asarray(y).ravel()
    yhat = np.asarray(yhat).ravel()
    return np.sqrt(np.mean((y - yhat)**2))

def r2(y, yhat):
    y = np.asarray(y).ravel()
    yhat = np.asarray(yhat).ravel()
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - y.mean())**2)
    return np.nan if ss_tot == 0 else 1 - ss_res / ss_tot

def tstat(x):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    return np.nan if len(x) < 2 else x.mean() / (x.std(ddof=1) / np.sqrt(len(x)))

In [40]:
def oos_metrics(y_true_folds, y_pred_folds, return_folds=False):
    ic_list   = [ic(y, yhat)   for y, yhat in zip(y_true_folds, y_pred_folds)]
    rmse_list = [rmse(y, yhat) for y, yhat in zip(y_true_folds, y_pred_folds)]
    r2_list   = [r2(y, yhat)   for y, yhat in zip(y_true_folds, y_pred_folds)]

    res = {
        "mean_oos_ic": np.nanmean(ic_list),
        "tstat_ic": tstat(ic_list),
        "mean_oos_rmse": np.nanmean(rmse_list),
        "mean_oos_r2": np.nanmean(r2_list),
    }

    if return_folds:
        res.update({
            "ic_folds": ic_list,
            "rmse_folds": rmse_list,
            "r2_folds": r2_list,
        })

    return res

Mean Forecast (0)

In [34]:
def zero_baseline(splits):
    y_true_folds = []
    y_pred_folds = []

    for fold in splits:
        y = fold["Y_test"].values.ravel()
        yhat = np.zeros_like(y)
        y_true_folds.append(y)
        y_pred_folds.append(yhat)

    return oos_metrics(y_true_folds, y_pred_folds)

In [35]:
res1 = zero_baseline(splits1)
res5 = zero_baseline(splits5)
res20 = zero_baseline(splits20)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/var/folders/43/y646xgv56z72xq7sm281s4zr0000gn/T/ipykernel_23342/3454508213.py:7: RuntimeWarning: Mean of empty slice
  "mean_oos_ic": np.nanmean(ic_list),


In [41]:
res1 = zero_baseline(splits1)
res5 = zero_baseline(splits5)
res20 = zero_baseline(splits20)

results = pd.DataFrame({
    "h=1": res1,
    "h=5": res5,
    "h=20": res20,
}).T

display(results.round(6))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/var/folders/43/y646xgv56z72xq7sm281s4zr0000gn/T/ipykernel_23342/3052757927.py:7: RuntimeWarning: Mean of empty slice
  "mean_oos_ic": np.nanmean(ic_list),


,mean_oos_ic,tstat_ic,mean_oos_rmse,mean_oos_r2
h=1,NaN,NaN,0.014553,-0.014521
h=5,NaN,NaN,0.031467,-0.096768
h=20,NaN,NaN,0.058803,-0.614936


ridge

In [44]:
import optuna
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
def ridge_oos(splits, val_size=252, n_trials=30, seed=42):
    y_true_folds = []
    y_pred_folds = []

    for fold in splits:
        X_train = fold["X_train"]
        Y_train = fold["Y_train"].values.ravel()
        X_test  = fold["X_test"]
        Y_test  = fold["Y_test"].values.ravel()

        X_tr = X_train.iloc[:-val_size]
        y_tr = Y_train[:-val_size]
        X_val = X_train.iloc[-val_size:]
        y_val = Y_train[-val_size:]

        def objective(trial):
            alpha = trial.suggest_float("alpha", 1e-4, 1e4, log=True)

            scaler = StandardScaler()
            X_tr_s = scaler.fit_transform(X_tr)
            X_val_s = scaler.transform(X_val)

            model = Ridge(alpha=alpha)
            model.fit(X_tr_s, y_tr)
            y_val_pred = model.predict(X_val_s)

            return rmse(y_val, y_val_pred)

        study = optuna.create_study(direction="minimize",
                                    sampler=optuna.samplers.TPESampler(seed=seed))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

        best_alpha = study.best_params["alpha"]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        model = Ridge(alpha=best_alpha)
        model.fit(X_train_s, Y_train)
        y_pred = model.predict(X_test_s)

        y_true_folds.append(Y_test)
        y_pred_folds.append(y_pred)

    return oos_metrics(y_true_folds, y_pred_folds)

In [45]:
res_ridge_1 = ridge_oos(splits1, val_size=252, n_trials=30)
res_ridge_5 = ridge_oos(splits5, val_size=252, n_trials=30)
res_ridge_20 = ridge_oos(splits20, val_size=252, n_trials=30)

[I 2026-04-19 14:57:19,616] A new study created in memory with name: no-name-213b3888-6e9b-43b5-b368-654e66508bf8
[I 2026-04-19 14:57:19,655] Trial 0 finished with value: 0.015178527717396601 and parameters: {'alpha': 0.09915644566638405}. Best is trial 0 with value: 0.015178527717396601.
[I 2026-04-19 14:57:19,660] Trial 1 finished with value: 0.014338403783346804 and parameters: {'alpha': 4033.800832600386}. Best is trial 1 with value: 0.014338403783346804.
[I 2026-04-19 14:57:19,664] Trial 2 finished with value: 0.015149223085372214 and parameters: {'alpha': 71.77141927992028}. Best is trial 1 with value: 0.014338403783346804.
[I 2026-04-19 14:57:19,667] Trial 3 finished with value: 0.015194166555354634 and parameters: {'alpha': 6.155564318973023}. Best is trial 1 with value: 0.014338403783346804.
[I 2026-04-19 14:57:19,670] Trial 4 finished with value: 0.015178870440677276 and parameters: {'alpha': 0.001770716864353783}. Best is trial 1 with value: 0.014338403783346804.
[I 2026-04-

In [46]:
results_ridge = pd.DataFrame({
    "h=1": res_ridge_1,
    "h=5": res_ridge_5,
    "h=20": res_ridge_20,
}).T

display(results_ridge.round(6))

,mean_oos_ic,tstat_ic,mean_oos_rmse,mean_oos_r2
h=1,0.025696,1.346186,0.014576,-0.018322
h=5,0.308879,11.031029,0.030568,-0.090637
h=20,0.421551,7.587647,0.050423,-0.208463


PCA

In [92]:
from sklearn.decomposition import PCA


def pca_oos_table(splits, n_components_list):
    rows = []

    for n_components in n_components_list:
        ev_folds = []

        for fold in splits:
            X_train = fold["X_train"]
            X_test  = fold["X_test"]

            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s  = scaler.transform(X_test)

            pca = PCA(n_components=n_components)
            pca.fit(X_train_s)

            Z_test = pca.transform(X_test_s)
            X_test_hat = Z_test @ pca.components_

            ss_total = np.sum(X_test_s ** 2)
            ss_resid = np.sum((X_test_s - X_test_hat) ** 2)

            ev_folds.append(1 - ss_resid / ss_total)

        rows.append({
            "n_components": n_components,
            "mean_oos_explained_variance": np.mean(ev_folds),
        })

    return pd.DataFrame(rows)

In [93]:
tab1 = pca_oos_table(splits1,  [1, 2, 3, 5, 10, 20])
tab5 = pca_oos_table(splits5,  [1, 2, 3, 5, 10, 20])
tab20 = pca_oos_table(splits20, [1, 2, 3, 5, 10, 20])

In [94]:
display(tab1.round(4))
display(tab5.round(4))
display(tab20.round(4))

,n_components,mean_oos_explained_variance
0,1,0.2486
1,2,0.3671
2,3,0.4571
3,5,0.5608
4,10,0.7067
5,20,0.9963


,n_components,mean_oos_explained_variance
0,1,0.2856
1,2,0.4369
2,3,0.5042
3,5,0.6039
4,10,0.7352
5,20,0.9992


,n_components,mean_oos_explained_variance
0,1,0.2804
1,2,0.4468
2,3,0.5130
3,5,0.6220
4,10,0.7577
5,20,0.9998


In [96]:
k = 10
def compute_pca_summary(splits, n_components=k):
    all_loadings = []
    explained_variance = []

    for fold in splits:
        X_train = fold["X_train"]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)

        pca = PCA(n_components=n_components)
        pca.fit(X_train_s)

        loadings = pd.DataFrame(pca.components_.T,
                                index=X_train.columns,
                                columns=[f'PC{i+1}' for i in range(n_components)])
        all_loadings.append(loadings)
        explained_variance.append(pca.explained_variance_ratio_)

    avg_loadings = sum(all_loadings) / len(all_loadings)
    avg_explained_variance = np.mean(explained_variance, axis=0)

    return avg_loadings, avg_explained_variance

def print_top_loadings(loadings, ev, horizon_name):
    print(f"\nHorizon: {horizon_name}")
    print("Explained variance ratio per component:")
    for i, ev_ratio in enumerate(ev):
        print(f"  PC{i+1}: {ev_ratio:.3f}")

    print("\nTop 5 variables per component:")
    for pc in loadings.columns:
        top_vars = loadings[pc].abs().sort_values(ascending=False).head(5)
        print(f"{pc}: {', '.join(top_vars.index)}")
loadings1, ev1 = compute_pca_summary(splits1)
loadings5, ev5 = compute_pca_summary(splits5)
loadings20, ev20 = compute_pca_summary(splits20)

print_top_loadings(loadings1, ev1, "1-step")
print_top_loadings(loadings5, ev5, "5-step")
print_top_loadings(loadings20, ev20, "20-step")


Horizon: 1-step
Explained variance ratio per component:
  PC1: 0.293
  PC2: 0.112
  PC3: 0.072
  PC4: 0.055
  PC5: 0.050
  PC6: 0.040
  PC7: 0.039
  PC8: 0.035
  PC9: 0.035
  PC10: 0.034

Top 5 variables per component:
PC1: AUDUSD=X, AUDJPY=X, NZDJPY=X, NZDUSD=X, EURJPY=X
PC2: USDCHF=X, CADCHF=X, EURCHF=X, USDJPY=X, CADJPY=X
PC3: CHFJPY=X, USDJPY=X, EURCHF=X, EURJPY=X, CADCHF=X
PC4: AUDCAD=X, AUDNZD=X, EURGBP=X, EURUSD=X, EURCHF=X
PC5: EURGBP=X, AUDNZD=X, AUDCAD=X, GBPUSD=X, EURCHF=X
PC6: AUDNZD=X, NZDUSD=X, AUDCAD=X, NZDJPY=X, DX-Y.NYB
PC7: AUDNZD=X, DX-Y.NYB, NZDUSD=X, NZDJPY=X, AUDCAD=X
PC8: USDPEN=X, USDCOP=X, USDCNY=X, USDRUB=X, USDBRL=X
PC9: USDRUB=X, USDPEN=X, USDCLP=X, USDCNY=X, USDCOP=X
PC10: USDCOP=X, USDCLP=X, USDRUB=X, EURUSD=X, USDCNY=X

Horizon: 5-step
Explained variance ratio per component:
  PC1: 0.329
  PC2: 0.129
  PC3: 0.078
  PC4: 0.054
  PC5: 0.050
  PC6: 0.042
  PC7: 0.040
  PC8: 0.035
  PC9: 0.034
  PC10: 0.034

Top 5 variables per component:
PC1: AUDUSD=X, NZDU

In [56]:
def pca_ridge_oos(splits, n_components=10, val_size=252, n_trials=30, seed=42):
    y_true_folds = []
    y_pred_folds = []
    ev_folds = []

    for fold in splits:
        X_train = fold["X_train"]
        Y_train = fold["Y_train"].values.ravel()
        X_test  = fold["X_test"]
        Y_test  = fold["Y_test"].values.ravel()

        X_tr  = X_train.iloc[:-val_size]
        y_tr  = Y_train[:-val_size]
        X_val = X_train.iloc[-val_size:]
        y_val = Y_train[-val_size:]

        def objective(trial):
            alpha = trial.suggest_float("alpha", 1e-4, 1e4, log=True)

            scaler = StandardScaler()
            X_tr_s  = scaler.fit_transform(X_tr)
            X_val_s = scaler.transform(X_val)

            pca = PCA(n_components=n_components)
            Z_tr  = pca.fit_transform(X_tr_s)
            Z_val = pca.transform(X_val_s)

            model = Ridge(alpha=alpha)
            model.fit(Z_tr, y_tr)
            y_val_pred = model.predict(Z_val)

            return rmse(y_val, y_val_pred)

        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=seed)
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

        best_alpha = study.best_params["alpha"]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s  = scaler.transform(X_test)

        pca = PCA(n_components=n_components)
        Z_train = pca.fit_transform(X_train_s)
        Z_test  = pca.transform(X_test_s)

        model = Ridge(alpha=best_alpha)
        model.fit(Z_train, Y_train)
        y_pred = model.predict(Z_test)

        X_test_hat = Z_test @ pca.components_
        ss_total = np.sum(X_test_s ** 2)
        ss_resid = np.sum((X_test_s - X_test_hat) ** 2)
        ev_folds.append(1 - ss_resid / ss_total)

        y_true_folds.append(Y_test)
        y_pred_folds.append(y_pred)

    res = oos_metrics(y_true_folds, y_pred_folds)
    res["mean_oos_pca_explained_variance"] = np.mean(ev_folds)
    res["pca_explained_variance_folds"] = ev_folds
    return res

In [57]:
res_pca_ridge_1  = pca_ridge_oos(splits1,  n_components=10, val_size=252, n_trials=30)
res_pca_ridge_5  = pca_ridge_oos(splits5,  n_components=10, val_size=252, n_trials=30)
res_pca_ridge_20 = pca_ridge_oos(splits20, n_components=10, val_size=252, n_trials=30)

[I 2026-04-19 15:10:02,188] A new study created in memory with name: no-name-f222811d-3774-4cbc-94a8-e9e0865ebccc
[I 2026-04-19 15:10:02,240] Trial 0 finished with value: 0.014503624265445084 and parameters: {'alpha': 0.09915644566638405}. Best is trial 0 with value: 0.014503624265445084.
[I 2026-04-19 15:10:02,243] Trial 1 finished with value: 0.014276982431990609 and parameters: {'alpha': 4033.800832600386}. Best is trial 1 with value: 0.014276982431990609.
[I 2026-04-19 15:10:02,246] Trial 2 finished with value: 0.014489985117168577 and parameters: {'alpha': 71.77141927992028}. Best is trial 1 with value: 0.014276982431990609.
[I 2026-04-19 15:10:02,249] Trial 3 finished with value: 0.014502425998595284 and parameters: {'alpha': 6.155564318973023}. Best is trial 1 with value: 0.014276982431990609.
[I 2026-04-19 15:10:02,252] Trial 4 finished with value: 0.014503643605233466 and parameters: {'alpha': 0.001770716864353783}. Best is trial 1 with value: 0.014276982431990609.
[I 2026-04-

In [59]:
results_pca_ridge = pd.DataFrame({
    "h=1":  res_pca_ridge_1,
    "h=5":  res_pca_ridge_5,
    "h=20": res_pca_ridge_20,
}).T
display(results_pca_ridge.drop(columns=["pca_explained_variance_folds"], errors="ignore").round(6))

,mean_oos_ic,tstat_ic,mean_oos_rmse,mean_oos_r2,mean_oos_pca_explained_variance
h=1,0.015835,0.766141,0.014567,-0.016966,0.706731
h=5,0.287368,10.220156,0.029731,0.003106,0.735228
h=20,0.425698,7.838713,0.050276,-0.200333,0.757687


PLS

In [61]:
from sklearn.cross_decomposition import PLSRegression
def pls_oos_table(splits, n_components_list):
    rows = []

    for n_components in n_components_list:
        ev_folds = []

        for fold in splits:
            X_train = fold["X_train"]
            Y_train = fold["Y_train"].values.ravel()
            X_test  = fold["X_test"]

            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s  = scaler.transform(X_test)

            pls = PLSRegression(n_components=n_components)
            pls.fit(X_train_s, Y_train)

            T_test = pls.transform(X_test_s)
            X_test_hat = T_test @ pls.x_loadings_.T

            ss_total = np.sum(X_test_s ** 2)
            ss_resid = np.sum((X_test_s - X_test_hat) ** 2)

            ev_folds.append(1 - ss_resid / ss_total)

        rows.append({
            "n_components": n_components,
            "mean_oos_explained_variance": np.mean(ev_folds),
        })

    return pd.DataFrame(rows)

In [62]:
tab_pls_1 = pls_oos_table(splits1, [1, 2, 3, 5, 10, 15, 20])
tab_pls_5 = pls_oos_table(splits5, [1, 2, 3, 5, 10, 15, 20])
tab_pls_20 = pls_oos_table(splits20, [1, 2, 3, 5, 10, 15, 20])

display(tab_pls_1.round(4))
display(tab_pls_5.round(4))
display(tab_pls_20.round(4))

,n_components,mean_oos_explained_variance
0,1,0.2126
1,2,0.3372
2,3,0.4123
3,5,0.5162
4,10,0.6793
5,15,0.7998
6,20,0.8885


,n_components,mean_oos_explained_variance
0,1,0.2801
1,2,0.3983
2,3,0.4709
3,5,0.5510
4,10,0.7083
5,15,0.8429
6,20,0.9654


,n_components,mean_oos_explained_variance
0,1,0.2729
1,2,0.4096
2,3,0.4929
3,5,0.5856
4,10,0.7702
5,15,0.9159
6,20,0.9840


In [63]:
res_pls_1  = pls_oos(splits1,  n_components=10)
res_pls_5  = pls_oos(splits5,  n_components=10)
res_pls_20 = pls_oos(splits20, n_components=10)

In [64]:
results_pls = pd.DataFrame({
    "h=1":  res_pls_1,
    "h=5":  res_pls_5,
    "h=20": res_pls_20,
}).T

display(results_pls.round(6))

,mean_oos_ic,tstat_ic,mean_oos_rmse,mean_oos_r2
h=1,0.011292,0.592072,0.014992,-0.102011
h=5,0.313119,11.156461,0.029899,-0.028231
h=20,0.404086,6.928618,0.052514,-0.320116


MLP autoencoder

In [68]:
from sklearn.neural_network import MLPRegressor

def ae_oos_table(splits, latent_dims):
    rows = []

    for k in latent_dims:
        ev_folds = []

        for f in splits:
            X_tr = f["X_train"]
            X_te = f["X_test"]

            sc = StandardScaler()
            X_tr_s = sc.fit_transform(X_tr)
            X_te_s = sc.transform(X_te)
            ae = MLPRegressor(
    hidden_layer_sizes=(50, k, 50),
    activation="relu",
    max_iter=200,
    random_state=42
)
            ae.fit(X_tr_s, X_tr_s)
            X_hat = ae.predict(X_te_s)

            ss_total = np.sum(X_te_s ** 2)
            ss_resid = np.sum((X_te_s - X_hat) ** 2)

            ev_folds.append(1 - ss_resid / ss_total)

        rows.append({
            "latent_dim": k,
            "mean_oos_explained_variance": np.mean(ev_folds),
        })

    return pd.DataFrame(rows)

In [69]:
tab_ae_1  = ae_oos_table(splits1,  [2, 5, 10, 20])
tab_ae_5  = ae_oos_table(splits5,  [2, 5, 10, 20])
tab_ae_20 = ae_oos_table(splits20, [2, 5, 10, 20])

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iteration

In [70]:
display(tab_ae_1.round(4))
display(tab_ae_5.round(4))
display(tab_ae_20.round(4))

,latent_dim,mean_oos_explained_variance
0,2,0.3856
1,5,0.5698
2,10,0.8049
3,20,0.9761


,latent_dim,mean_oos_explained_variance
0,2,0.4387
1,5,0.6020
2,10,0.8477
3,20,0.9832


,latent_dim,mean_oos_explained_variance
0,2,0.4236
1,5,0.6019
2,10,0.8342
3,20,0.9787


In [71]:
def ae_ridge_oos(splits, k=10, val_size=252, n_trials=30, seed=42):
    y_true_folds = []
    y_pred_folds = []

    for f in splits:
        X_train = f["X_train"]
        Y_train = f["Y_train"].values.ravel()
        X_test  = f["X_test"]
        Y_test  = f["Y_test"].values.ravel()

        X_tr  = X_train.iloc[:-val_size]
        y_tr  = Y_train[:-val_size]
        X_val = X_train.iloc[-val_size:]
        y_val = Y_train[-val_size:]

        def relu(x):
            return np.maximum(0, x)

        def encode_middle(ae, X):
            H1 = relu(X @ ae.coefs_[0] + ae.intercepts_[0])
            Z  = relu(H1 @ ae.coefs_[1] + ae.intercepts_[1])   
            return Z

        def objective(trial):
            alpha = trial.suggest_float("alpha", 1e-4, 1e4, log=True)

            sc = StandardScaler()
            X_tr_s  = sc.fit_transform(X_tr)
            X_val_s = sc.transform(X_val)

            ae = MLPRegressor(
                hidden_layer_sizes=(50, k, 50),
                activation="relu",
                max_iter=200,
                random_state=seed
            )
            ae.fit(X_tr_s, X_tr_s)

            Z_tr  = encode_middle(ae, X_tr_s)
            Z_val = encode_middle(ae, X_val_s)

            model = Ridge(alpha=alpha)
            model.fit(Z_tr, y_tr)
            y_val_pred = model.predict(Z_val)

            return rmse(y_val, y_val_pred)

        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=seed)
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
        best_alpha = study.best_params["alpha"]

        sc = StandardScaler()
        X_train_s = sc.fit_transform(X_train)
        X_test_s  = sc.transform(X_test)

        ae = MLPRegressor(
            hidden_layer_sizes=(50, k, 50),
            activation="relu",
            max_iter=200,
            random_state=seed
        )
        ae.fit(X_train_s, X_train_s)

        Z_train = encode_middle(ae, X_train_s)
        Z_test  = encode_middle(ae, X_test_s)

        model = Ridge(alpha=best_alpha)
        model.fit(Z_train, Y_train)
        y_pred = model.predict(Z_test)

        y_true_folds.append(Y_test)
        y_pred_folds.append(y_pred)

    return oos_metrics(y_true_folds, y_pred_folds)

In [72]:
res_ae_ridge_1  = ae_ridge_oos(splits1,  k=10, val_size=252, n_trials=30)
res_ae_ridge_5  = ae_ridge_oos(splits5,  k=10, val_size=252, n_trials=30)
res_ae_ridge_20 = ae_ridge_oos(splits20, k=10, val_size=252, n_trials=30)

[I 2026-04-19 15:48:15,842] A new study created in memory with name: no-name-4c719640-f592-440a-8a1e-dc265afb249f
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2026-04-19 15:48:16,450] Trial 0 finished with value: 0.014668791876674584 and parameters: {'alpha': 0.09915644566638405}. Best is trial 0 with value: 0.014668791876674584.
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2026-04-19 15:48:17,032] Trial 1 finished with value: 0.014232399593830766 and parameters: {'alpha': 4033.800832600386}. Best is trial 1 with value: 0.014232399593830766

In [73]:
results_ae_ridge = pd.DataFrame({
    "h=1": res_ae_ridge_1,
    "h=5": res_ae_ridge_5,
    "h=20": res_ae_ridge_20,
}).T
display(results_ae_ridge.round(6))

,mean_oos_ic,tstat_ic,mean_oos_rmse,mean_oos_r2
h=1,-0.004341,-0.226999,0.014774,-0.055886
h=5,0.284496,9.495519,0.029861,-0.008539
h=20,0.400924,6.669643,0.051017,-0.239910


comparison

In [77]:
def clean(res, drop_cols=None):
    if drop_cols is None:
        drop_cols = []
    return {k: v for k, v in res.items() if k not in drop_cols}


rows = []

for h, res_ae, res_pca, res_pls, res_ridge, res_zero in [
    (1,  res_ae_ridge_1,  res_pca_ridge_1,  res_pls_1,  res_ridge_1,  res1),
    (5,  res_ae_ridge_5,  res_pca_ridge_5,  res_pls_5,  res_ridge_5,  res5),
    (20, res_ae_ridge_20, res_pca_ridge_20, res_pls_20, res_ridge_20, res20),
]:
    rows.append({"model": "AE+Ridge", "h": h, **clean(res_ae)})
    rows.append({"model": "PCA+Ridge", "h": h, **clean(res_pca, ["pca_explained_variance_folds"])})
    rows.append({"model": "PLS", "h": h, **clean(res_pls)})
    rows.append({"model": "Ridge", "h": h, **clean(res_ridge)})
    rows.append({"model": "Zero", "h": h, **clean(res_zero)})

comparison = pd.DataFrame(rows)
comparison = comparison.set_index(["model", "h"]).sort_index()

display(comparison.round(6))

mean_oos_ic   tstat_ic  mean_oos_rmse  mean_oos_r2  \
model     h                                                        
AE+Ridge  1     -0.004341  -0.226999       0.014774    -0.055886   
          5      0.284496   9.495519       0.029861    -0.008539   
          20     0.400924   6.669643       0.051017    -0.239910   
PCA+Ridge 1      0.015835   0.766141       0.014567    -0.016966   
          5      0.287368  10.220156       0.029731     0.003106   
          20     0.425698   7.838713       0.050276    -0.200333   
PLS       1      0.011292   0.592072       0.014992    -0.102011   
          5      0.313119  11.156461       0.029899    -0.028231   
          20     0.404086   6.928618       0.052514    -0.320116   
Ridge     1      0.025696   1.346186       0.014576    -0.018322   
          5      0.308879  11.031029       0.030568    -0.090637   
          20     0.421551   7.587647       0.050423    -0.208463   
Zero      1           NaN        NaN       0.014553    -0.014521   
          5           NaN        NaN       0.031467    -0.096768   
          20          NaN        NaN       0.058803    -0.614936   

              mean_oos_pca_explained_variance  
model     h                                    
AE+Ridge  1                               NaN  
          5                               NaN  
          20                              NaN  
PCA+Ridge 1                          0.706731  
          5                          0.735228  
          20                         0.757687  
PLS       1                               NaN  
          5                               NaN  
          20                              NaN  
Ridge     1                               NaN  
          5                               NaN  
          20                              NaN  
Zero      1                               NaN  
          5                               NaN  
          20                              NaN